# Fixed 20-config benchmark Qwen / GLM

Финальный benchmark без adaptive selection. Конфиги зафиксированы на базе уже выполненного capability discovery из `qwen_glm_adaptive_benchmark_work_done.ipynb`.

Главный фокус — **время ответа**, особенно цена и длина reasoning/thinking части. Поэтому матрица расширена до 20 конфигов и отдельно варьирует:

- `reasoning/thinking off`;
- `reasoning/thinking on`;
- разные уровни effort: `none`, `low`, `medium`, `high`;
- `thinking.enabled`;
- `thinking.budget_tokens`;
- `include_reasoning`;
- `max_tokens 2048/4096`;
- sampling и `repetition_penalty` только как дополнительные latency-края.

Всего: 20 конфигов, 10 для Qwen и 10 для GLM. Порядок прогона остается question-major: каждый prompt сначала проходит по всем конфигам, потом следующий prompt.


## Что показал capability discovery

Доступные параметры:

**Qwen3.5-397b**
- `max_tokens`: 2048, 4096
- `temperature`: 0.2, 0.5, 0.7
- `top_p`: 0.8, 0.9, 0.95
- `repetition_penalty`: 1.05
- reasoning/thinking:
  - `reasoning_effort`: `none`, `low`, `medium`, `high`
  - `reasoning: {"effort": ...}`: `none`, `low`, `medium`, `high`
  - `reasoning: {"enabled": false/true}`
  - `enable_thinking`: `false/true`
  - `thinking: {"enabled": false/true}`
  - `thinking: {"budget_tokens": 512}`
  - `chat_template_kwargs: {"enable_thinking": false/true}`
  - `include_reasoning: true`

**glm-5.1**
- `max_tokens`: 2048, 4096
- `temperature`: 0.2, 0.5
- `top_p`: 0.8, 0.95
- `repetition_penalty`: 1.05
- `temperature=0.7` и `top_p=0.9` в capability discovery ушли в timeout, поэтому в финальные конфиги GLM не включены.
- reasoning/thinking: доступны те же семейства параметров, включая `reasoning_effort`, `reasoning: {"effort": ...}`, `enable_thinking`, `thinking.enabled`, `thinking.budget_tokens=512`, `chat_template_kwargs.enable_thinking`, `include_reasoning=true`.

Важное ограничение: discovery подтвердил саму доступность `thinking.budget_tokens` на значении `512`. Поэтому в основной матрице используется именно `512`, а не непроверенные значения `256/1024`, чтобы benchmark не превратился в повторный capability discovery.


In [ ]:
# =========================
# 0. Imports
# =========================

import csv
import hashlib
import json
import os
import random
import re
import sys
import time
import warnings
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", message="Unverified HTTPS request")
warnings.filterwarnings("ignore", category=UserWarning)

try:
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
except Exception:
    pass

try:
    from gigachat import GigaChat
except Exception as exc:
    raise RuntimeError("Не удалось импортировать gigachat. Запускайте в окружении, где работал ift_agent_test.ipynb") from exc

RUN_STARTED_AT = datetime.now().strftime("%Y%m%d_%H%M%S")
print("Run started:", RUN_STARTED_AT)


In [ ]:
# =========================
# 1. Settings
# =========================

DATA_CSV = Path("max_inputs_first_200_unique_questions.csv")

GIGACHAT_API_URL = "https://gigachat-ift.sberdevices.delta.sbrf.ru/v1"
SCOPE = "GIGACHAT_API_PERS"
CERT_FILE = "45500_cert.pem"
KEY_FILE = "45500_key.pem"
VERIFY_SSL_CERTS = False
PROFANITY_CHECK = False

RUN_BENCHMARK = True

# full — весь CSV; sample/smoke — быстрые режимы проверки.
RUN_MODE = "full"  # "full" | "sample" | "smoke"
SAMPLE_ROWS_TOTAL = 60
SMOKE_ROWS_PER_STAGE = 2
RANDOM_SEED = 42

MAX_WORKERS = 1
REQUEST_TIMEOUT_SEC = 270
MAX_RETRIES = 2
RETRY_BACKOFF_SEC = 8
JITTER_SEC = 2
USE_STREAMING = True
FALLBACK_TO_NON_STREAMING = True
CONFIG_ORDER_STRATEGY = "rotate"  # "fixed" | "rotate" | "shuffle_by_question"

SAVE_FULL_ANSWERS_TO_EXCEL = True
ANSWER_PREVIEW_CHARS = 500

# Перезаписываем checkpoint_latest.xlsx каждые 3 вопроса.
# Один вопрос = весь набор конфигов для одного prompt-а.
CHECKPOINT_EVERY_QUESTIONS = 3

# Стабильный label нужен для resume: если ноутбук упал или ядро перезапущено,
# тот же RUN_LABEL продолжит raw_results.jsonl и пропустит уже завершенные пары prompt × config.
# Для нового чистого прогона поменяйте RUN_LABEL, например на f"qwen_glm_fixed20_{RUN_STARTED_AT}".
RUN_LABEL = "qwen_glm_fixed20_current"

OUTPUT_DIR = Path("benchmark_outputs") / RUN_LABEL
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_JSONL_PATH = OUTPUT_DIR / "raw_results.jsonl"
EXCEL_REPORT_PATH = OUTPUT_DIR / "qwen_glm_fixed20_benchmark_report.xlsx"
CHECKPOINT_LATEST_PATH = OUTPUT_DIR / "checkpoint_latest.xlsx"

print("Output dir:", OUTPUT_DIR.resolve())
print("Raw JSONL:", RAW_JSONL_PATH.resolve())


In [ ]:
# =========================
# 2. Load dataset
# =========================

csv.field_size_limit(sys.maxsize)
with DATA_CSV.open("r", encoding="utf-8-sig", newline="") as f:
    df_inputs = pd.DataFrame(list(csv.DictReader(f)))

required = ["type", "stage", "question", "system_prompt", "user_prompt"]
missing = [c for c in required if c not in df_inputs.columns]
if missing:
    raise ValueError(f"Нет обязательных колонок: {missing}")

if "llm_input_human_json" in df_inputs.columns:
    print("llm_input_human_json найден, но в запросы не используется.")

df_inputs = df_inputs.reset_index(drop=True).copy()
df_inputs["dataset_row_id"] = df_inputs.index.astype(int)
df_inputs["system_chars"] = df_inputs["system_prompt"].fillna("").str.len()
df_inputs["user_chars"] = df_inputs["user_prompt"].fillna("").str.len()
df_inputs["input_chars"] = df_inputs["system_chars"] + df_inputs["user_chars"]
df_inputs["old_output_chars"] = df_inputs.get("llm_output", pd.Series([""] * len(df_inputs))).fillna("").str.len()

display(df_inputs.groupby("type").agg(rows=("dataset_row_id", "count"), min_chars=("input_chars", "min"), p50_chars=("input_chars", "median"), max_chars=("input_chars", "max")).reset_index())


In [ ]:
# =========================
# 3. Available parameters summary from done discovery
# =========================

AVAILABLE_PARAMETERS = [
    {"model_key": "qwen", "model": "Qwen3.5-397b", "parameter": "max_tokens", "available_values": "2048, 4096", "used_in_configs": "2048, 4096", "note": "Прямо влияет на верхнюю границу генерации, обрезания и latency."},
    {"model_key": "qwen", "model": "Qwen3.5-397b", "parameter": "temperature", "available_values": "0.2, 0.5, 0.7", "used_in_configs": "0.2, 0.7", "note": "Sampling; 0.2 как стабильный режим, 0.7 как верхняя проверка."},
    {"model_key": "qwen", "model": "Qwen3.5-397b", "parameter": "top_p", "available_values": "0.8, 0.9, 0.95", "used_in_configs": "0.8, 0.9, 0.95", "note": "Sampling; влияет на вариативность и потенциально длину/скорость ответа."},
    {"model_key": "qwen", "model": "Qwen3.5-397b", "parameter": "repetition_penalty", "available_values": "1.05", "used_in_configs": "1.05", "note": "Может снижать повторы и зацикливание на длинном контексте."},
    {"model_key": "qwen", "model": "Qwen3.5-397b", "parameter": "reasoning_effort", "available_values": "none, low, medium, high", "used_in_configs": "none/low/medium/high via reasoning object", "note": "Главная ось теста: сколько времени добавляет разная глубина reasoning."},
    {"model_key": "qwen", "model": "Qwen3.5-397b", "parameter": "enable_thinking", "available_values": "false, true", "used_in_configs": "not used directly", "note": "Доступен, но для Qwen основной контроль сделан через reasoning.effort, чтобы не смешивать механизмы."},
    {"model_key": "qwen", "model": "Qwen3.5-397b", "parameter": "thinking.enabled", "available_values": "false, true", "used_in_configs": "false, true", "note": "Проверяет альтернативное прямое выключение/включение thinking без effort."},
    {"model_key": "qwen", "model": "Qwen3.5-397b", "parameter": "thinking.budget_tokens", "available_values": "512 confirmed", "used_in_configs": "512", "note": "Проверяет ограниченный budget reasoning/thinking части."},
    {"model_key": "qwen", "model": "Qwen3.5-397b", "parameter": "chat_template_kwargs.enable_thinking", "available_values": "false, true", "used_in_configs": "not used in top 20", "note": "Доступен, но в лимите 10 конфигов на Qwen уступил место прямому thinking.enabled и thinking budget."},
    {"model_key": "qwen", "model": "Qwen3.5-397b", "parameter": "include_reasoning", "available_values": "true", "used_in_configs": "true with medium reasoning", "note": "Проверяет, увеличивает ли логирование reasoning видимую длину/ответ/метрики."},

    {"model_key": "glm", "model": "glm-5.1", "parameter": "max_tokens", "available_values": "2048, 4096", "used_in_configs": "2048, 4096", "note": "Прямо влияет на верхнюю границу генерации, обрезания и latency."},
    {"model_key": "glm", "model": "glm-5.1", "parameter": "temperature", "available_values": "0.2, 0.5; 0.7 timeout", "used_in_configs": "0.2, 0.5", "note": "0.7 исключен как нестабильный по capability discovery."},
    {"model_key": "glm", "model": "glm-5.1", "parameter": "top_p", "available_values": "0.8, 0.95; 0.9 timeout", "used_in_configs": "0.8, 0.95", "note": "0.9 исключен как нестабильный по capability discovery."},
    {"model_key": "glm", "model": "glm-5.1", "parameter": "repetition_penalty", "available_values": "1.05", "used_in_configs": "1.05", "note": "Может снижать повторы и зацикливание на длинном контексте."},
    {"model_key": "glm", "model": "glm-5.1", "parameter": "reasoning_effort", "available_values": "none, low, medium, high", "used_in_configs": "none, low, medium, high", "note": "Главная ось теста: сколько времени добавляет разная глубина reasoning."},
    {"model_key": "glm", "model": "glm-5.1", "parameter": "enable_thinking", "available_values": "false, true", "used_in_configs": "false, true", "note": "Основной бинарный переключатель thinking для GLM."},
    {"model_key": "glm", "model": "glm-5.1", "parameter": "thinking.enabled", "available_values": "false, true", "used_in_configs": "true", "note": "Проверяет альтернативное прямое включение thinking без effort."},
    {"model_key": "glm", "model": "glm-5.1", "parameter": "thinking.budget_tokens", "available_values": "512 confirmed", "used_in_configs": "512", "note": "Проверяет ограниченный budget reasoning/thinking части."},
    {"model_key": "glm", "model": "glm-5.1", "parameter": "chat_template_kwargs.enable_thinking", "available_values": "false, true", "used_in_configs": "not used in top 20", "note": "Доступен, но уступил место effort/budget/include_reasoning в лимите 10 конфигов на модель."},
    {"model_key": "glm", "model": "glm-5.1", "parameter": "include_reasoning", "available_values": "true", "used_in_configs": "true with enable_thinking", "note": "Проверяет, увеличивает ли логирование reasoning видимую длину/ответ/метрики."},
]

df_available_parameters = pd.DataFrame(AVAILABLE_PARAMETERS)
display(df_available_parameters)


In [ ]:
# =========================
# 4. Fixed 20 configs
# =========================

FIXED_CONFIGS = [
    # Qwen: стабильная база + effort ladder + thinking-specific controls.
    {
        "config_id": "qwen_r_none_4096",
        "model_key": "qwen", "model": "Qwen3.5-397b",
        "max_tokens": 4096, "temperature": 0.2, "top_p": 0.9, "repetition_penalty": None,
        "extra_params": {"reasoning": {"effort": "none"}},
        "reasoning_mode": "off", "reasoning_control": "reasoning.effort=none",
        "why": "Базовый Qwen без reasoning: контроль скорости и качества.",
    },
    {
        "config_id": "qwen_r_low_4096",
        "model_key": "qwen", "model": "Qwen3.5-397b",
        "max_tokens": 4096, "temperature": 0.2, "top_p": 0.9, "repetition_penalty": None,
        "extra_params": {"reasoning": {"effort": "low"}},
        "reasoning_mode": "on_low", "reasoning_control": "reasoning.effort=low",
        "why": "Минимальный включенный reasoning: кандидат на сокращение reasoning-части.",
    },
    {
        "config_id": "qwen_r_medium_4096",
        "model_key": "qwen", "model": "Qwen3.5-397b",
        "max_tokens": 4096, "temperature": 0.2, "top_p": 0.9, "repetition_penalty": None,
        "extra_params": {"reasoning": {"effort": "medium"}},
        "reasoning_mode": "on_medium", "reasoning_control": "reasoning.effort=medium",
        "why": "Средний reasoning: основной качественный кандидат.",
    },
    {
        "config_id": "qwen_r_high_4096",
        "model_key": "qwen", "model": "Qwen3.5-397b",
        "max_tokens": 4096, "temperature": 0.2, "top_p": 0.9, "repetition_penalty": None,
        "extra_params": {"reasoning": {"effort": "high"}},
        "reasoning_mode": "on_high", "reasoning_control": "reasoning.effort=high",
        "why": "Верхний reasoning effort: измеряем максимальную цену рассуждения.",
    },
    {
        "config_id": "qwen_r_none_2048",
        "model_key": "qwen", "model": "Qwen3.5-397b",
        "max_tokens": 2048, "temperature": 0.2, "top_p": 0.9, "repetition_penalty": None,
        "extra_params": {"reasoning": {"effort": "none"}},
        "reasoning_mode": "off", "reasoning_control": "reasoning.effort=none + max_tokens=2048",
        "why": "Нижний лимит без reasoning: проверка ускорения и риска обрезания.",
    },
    {
        "config_id": "qwen_r_low_2048",
        "model_key": "qwen", "model": "Qwen3.5-397b",
        "max_tokens": 2048, "temperature": 0.2, "top_p": 0.9, "repetition_penalty": None,
        "extra_params": {"reasoning": {"effort": "low"}},
        "reasoning_mode": "on_low", "reasoning_control": "reasoning.effort=low + max_tokens=2048",
        "why": "Попытка сохранить немного reasoning при коротком лимите ответа.",
    },
    {
        "config_id": "qwen_thinking_disabled_4096",
        "model_key": "qwen", "model": "Qwen3.5-397b",
        "max_tokens": 4096, "temperature": 0.2, "top_p": 0.9, "repetition_penalty": None,
        "extra_params": {"thinking": {"enabled": False}},
        "reasoning_mode": "thinking_off", "reasoning_control": "thinking.enabled=false",
        "why": "Альтернативный способ выключить thinking, чтобы сравнить с reasoning.effort=none.",
    },
    {
        "config_id": "qwen_thinking_budget512_4096",
        "model_key": "qwen", "model": "Qwen3.5-397b",
        "max_tokens": 4096, "temperature": 0.2, "top_p": 0.9, "repetition_penalty": None,
        "extra_params": {"thinking": {"budget_tokens": 512}},
        "reasoning_mode": "thinking_budget", "reasoning_control": "thinking.budget_tokens=512",
        "why": "Проверка явного ограничения thinking budget: потенциальный способ сократить reasoning.",
    },
    {
        "config_id": "qwen_thinking_enabled_true_4096",
        "model_key": "qwen", "model": "Qwen3.5-397b",
        "max_tokens": 4096, "temperature": 0.2, "top_p": 0.9, "repetition_penalty": None,
        "extra_params": {"thinking": {"enabled": True}},
        "reasoning_mode": "thinking_on", "reasoning_control": "thinking.enabled=true",
        "why": "Прямое включение thinking через thinking.enabled=true.",
    },
    {
        "config_id": "qwen_include_reasoning_broad_4096",
        "model_key": "qwen", "model": "Qwen3.5-397b",
        "max_tokens": 4096, "temperature": 0.7, "top_p": 0.95, "repetition_penalty": 1.05,
        "extra_params": {"reasoning": {"effort": "medium"}, "include_reasoning": True},
        "reasoning_mode": "include_reasoning", "reasoning_control": "reasoning.effort=medium + include_reasoning=true",
        "why": "Проверка, меняет ли include_reasoning видимый reasoning/метрики на более вариативном Qwen.",
    },

    # GLM: enable_thinking + effort ladder + thinking budget/include_reasoning.
    {
        "config_id": "glm_enable_thinking_false_4096",
        "model_key": "glm", "model": "glm-5.1",
        "max_tokens": 4096, "temperature": 0.2, "top_p": 0.8, "repetition_penalty": None,
        "extra_params": {"enable_thinking": False},
        "reasoning_mode": "off", "reasoning_control": "enable_thinking=false",
        "why": "Базовый GLM без thinking.",
    },
    {
        "config_id": "glm_enable_thinking_true_4096",
        "model_key": "glm", "model": "glm-5.1",
        "max_tokens": 4096, "temperature": 0.2, "top_p": 0.8, "repetition_penalty": None,
        "extra_params": {"enable_thinking": True},
        "reasoning_mode": "on", "reasoning_control": "enable_thinking=true",
        "why": "Базовый GLM с thinking: пара к enable_thinking=false.",
    },
    {
        "config_id": "glm_reasoning_effort_none_4096",
        "model_key": "glm", "model": "glm-5.1",
        "max_tokens": 4096, "temperature": 0.2, "top_p": 0.8, "repetition_penalty": None,
        "extra_params": {"reasoning_effort": "none"},
        "reasoning_mode": "off", "reasoning_control": "reasoning_effort=none",
        "why": "Альтернативное выключение reasoning через effort=none.",
    },
    {
        "config_id": "glm_reasoning_effort_low_4096",
        "model_key": "glm", "model": "glm-5.1",
        "max_tokens": 4096, "temperature": 0.2, "top_p": 0.8, "repetition_penalty": None,
        "extra_params": {"reasoning_effort": "low"},
        "reasoning_mode": "on_low", "reasoning_control": "reasoning_effort=low",
        "why": "Минимальный effort: главный кандидат на сокращение reasoning-части GLM.",
    },
    {
        "config_id": "glm_reasoning_effort_medium_4096",
        "model_key": "glm", "model": "glm-5.1",
        "max_tokens": 4096, "temperature": 0.2, "top_p": 0.8, "repetition_penalty": None,
        "extra_params": {"reasoning_effort": "medium"},
        "reasoning_mode": "on_medium", "reasoning_control": "reasoning_effort=medium",
        "why": "Средний effort для GLM.",
    },
    {
        "config_id": "glm_reasoning_effort_high_4096",
        "model_key": "glm", "model": "glm-5.1",
        "max_tokens": 4096, "temperature": 0.2, "top_p": 0.8, "repetition_penalty": None,
        "extra_params": {"reasoning_effort": "high"},
        "reasoning_mode": "on_high", "reasoning_control": "reasoning_effort=high",
        "why": "Верхний effort: измеряем максимальную цену reasoning для GLM.",
    },
    {
        "config_id": "glm_enable_thinking_false_2048",
        "model_key": "glm", "model": "glm-5.1",
        "max_tokens": 2048, "temperature": 0.2, "top_p": 0.8, "repetition_penalty": None,
        "extra_params": {"enable_thinking": False},
        "reasoning_mode": "off", "reasoning_control": "enable_thinking=false + max_tokens=2048",
        "why": "Нижний лимит без thinking: быстрый, но рискованный режим.",
    },
    {
        "config_id": "glm_thinking_enabled_true_4096",
        "model_key": "glm", "model": "glm-5.1",
        "max_tokens": 4096, "temperature": 0.2, "top_p": 0.8, "repetition_penalty": None,
        "extra_params": {"thinking": {"enabled": True}},
        "reasoning_mode": "thinking_on", "reasoning_control": "thinking.enabled=true",
        "why": "Прямое включение thinking через thinking.enabled=true.",
    },
    {
        "config_id": "glm_thinking_budget512_4096",
        "model_key": "glm", "model": "glm-5.1",
        "max_tokens": 4096, "temperature": 0.2, "top_p": 0.8, "repetition_penalty": None,
        "extra_params": {"thinking": {"budget_tokens": 512}},
        "reasoning_mode": "thinking_budget", "reasoning_control": "thinking.budget_tokens=512",
        "why": "Проверка ограничения thinking budget на подтвержденном значении 512.",
    },
    {
        "config_id": "glm_include_reasoning_broad_4096",
        "model_key": "glm", "model": "glm-5.1",
        "max_tokens": 4096, "temperature": 0.5, "top_p": 0.95, "repetition_penalty": 1.05,
        "extra_params": {"enable_thinking": True, "include_reasoning": True},
        "reasoning_mode": "include_reasoning", "reasoning_control": "enable_thinking=true + include_reasoning=true",
        "why": "Проверка include_reasoning на самом широком стабильном GLM-конфиге.",
    },
]

df_configs = pd.DataFrame([
    {**{k: v for k, v in cfg.items() if k != "extra_params"}, "extra_params_json": json.dumps(cfg["extra_params"], ensure_ascii=False, sort_keys=True)}
    for cfg in FIXED_CONFIGS
])

display(df_configs[[
    "config_id", "model_key", "model", "reasoning_mode", "reasoning_control",
    "max_tokens", "temperature", "top_p", "repetition_penalty", "extra_params_json", "why"
]])
print("Configs:", len(df_configs))


In [ ]:
# =========================
# 5. Helpers
# =========================

_client_cache: Dict[Tuple[Any, ...], GigaChat] = {}


def make_client(timeout: int) -> GigaChat:
    if MAX_WORKERS and MAX_WORKERS > 1:
        return GigaChat(
            base_url=GIGACHAT_API_URL, scope=SCOPE, timeout=timeout,
            profanity_check=PROFANITY_CHECK, cert_file=CERT_FILE, key_file=KEY_FILE,
            verify_ssl_certs=VERIFY_SSL_CERTS,
        )
    key = (GIGACHAT_API_URL, SCOPE, CERT_FILE, KEY_FILE, timeout, VERIFY_SSL_CERTS, PROFANITY_CHECK)
    if key not in _client_cache:
        _client_cache[key] = GigaChat(
            base_url=GIGACHAT_API_URL, scope=SCOPE, timeout=timeout,
            profanity_check=PROFANITY_CHECK, cert_file=CERT_FILE, key_file=KEY_FILE,
            verify_ssl_certs=VERIFY_SSL_CERTS,
        )
    return _client_cache[key]


def obj_to_dict(obj: Any) -> Any:
    if obj is None or isinstance(obj, (str, int, float, bool)):
        return obj
    if isinstance(obj, list):
        return [obj_to_dict(x) for x in obj]
    if isinstance(obj, tuple):
        return [obj_to_dict(x) for x in obj]
    if isinstance(obj, dict):
        return {str(k): obj_to_dict(v) for k, v in obj.items()}
    if hasattr(obj, "model_dump"):
        try:
            return obj.model_dump(by_alias=True, exclude_none=False)
        except TypeError:
            return obj.model_dump()
    if hasattr(obj, "dict"):
        try:
            return obj.dict(by_alias=True)
        except TypeError:
            return obj.dict()
    return str(obj)


def deep_get(data: Any, paths: List[List[Any]], default=None):
    for path in paths:
        cur = data
        ok = True
        for key in path:
            try:
                cur = cur[key] if isinstance(key, int) else cur.get(key)
            except Exception:
                ok = False
                break
            if cur is None:
                ok = False
                break
        if ok:
            return cur
    return default


def deep_merge(base: Dict[str, Any], patch: Dict[str, Any]) -> Dict[str, Any]:
    out = dict(base)
    for k, v in (patch or {}).items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = deep_merge(out[k], v)
        else:
            out[k] = v
    return out


def text_hash(text: str) -> str:
    return hashlib.sha256((text or "").encode("utf-8")).hexdigest()[:16]


def extract_content(response_dict: Dict[str, Any]) -> str:
    v = deep_get(response_dict, [["choices", 0, "message", "content"]], "")
    return v if isinstance(v, str) else ""


def extract_finish_reason(response_dict: Dict[str, Any]) -> Optional[str]:
    return deep_get(response_dict, [["choices", 0, "finish_reason"], ["choices", 0, "finishReason"]])


def extract_reasoning_text(response_dict: Dict[str, Any]) -> str:
    candidates = [
        ["choices", 0, "message", "reasoning"], ["choices", 0, "message", "reasoning_content"],
        ["choices", 0, "message", "thinking"], ["choices", 0, "delta", "reasoning"],
        ["choices", 0, "delta", "reasoning_content"],
    ]
    v = deep_get(response_dict, candidates, "")
    return v if isinstance(v, str) else ""


def normalize_usage(usage: Any) -> Dict[str, Any]:
    d = obj_to_dict(usage) or {}
    return d if isinstance(d, dict) else {}


def extract_usage_numbers(usage: Dict[str, Any]) -> Dict[str, Any]:
    details = usage.get("completion_tokens_details") or usage.get("completionTokensDetails") or {}
    return {
        "prompt_tokens": usage.get("prompt_tokens") or usage.get("promptTokens"),
        "completion_tokens": usage.get("completion_tokens") or usage.get("completionTokens"),
        "total_tokens": usage.get("total_tokens") or usage.get("totalTokens"),
        "precached_prompt_tokens": usage.get("precached_prompt_tokens") or usage.get("precachedPromptTokens"),
        "reasoning_tokens": usage.get("reasoning_tokens") or usage.get("reasoningTokens") or details.get("reasoning_tokens") or details.get("reasoningTokens"),
    }


def count_words(text: str) -> int:
    return len(re.findall(r"[A-Za-zА-Яа-яЁё0-9_]+", text or ""))


In [ ]:
# =========================
# 6. LLM call using raw dict payload
# =========================


def build_payload(system_prompt: str, user_prompt: str, cfg: Dict[str, Any], stream: bool) -> Dict[str, Any]:
    base_params = {
        "max_tokens": int(cfg["max_tokens"]),
        "temperature": None if pd.isna(cfg.get("temperature")) else float(cfg.get("temperature")),
        "top_p": None if pd.isna(cfg.get("top_p")) else float(cfg.get("top_p")),
        "repetition_penalty": None if pd.isna(cfg.get("repetition_penalty")) else float(cfg.get("repetition_penalty")),
    }
    base_params = {k: v for k, v in base_params.items() if v is not None}
    extra_params = json.loads(cfg.get("extra_params_json") or "{}")
    payload = {
        "model": cfg["model"],
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "stream": bool(stream),
        "profanity_check": PROFANITY_CHECK,
    }
    payload = deep_merge(payload, base_params)
    payload = deep_merge(payload, extra_params)
    return payload


def run_payload(system_prompt: str, user_prompt: str, cfg: Dict[str, Any]) -> Dict[str, Any]:
    client = make_client(REQUEST_TIMEOUT_SEC)
    start = time.perf_counter()
    first_chunk_at = None
    first_content_at = None
    chunks_seen = 0
    content_parts: List[str] = []
    reasoning_parts: List[str] = []
    last_chunk_dict: Dict[str, Any] = {}
    response_dict: Dict[str, Any] = {}
    usage_dict: Dict[str, Any] = {}

    if USE_STREAMING:
        payload = build_payload(system_prompt, user_prompt, cfg, stream=True)
        try:
            for chunk in client.stream(payload):
                now = time.perf_counter()
                chunks_seen += 1
                if first_chunk_at is None:
                    first_chunk_at = now
                chunk_dict = obj_to_dict(chunk) or {}
                last_chunk_dict = chunk_dict
                delta_content = deep_get(chunk_dict, [["choices", 0, "delta", "content"]], "")
                if isinstance(delta_content, str) and delta_content:
                    if first_content_at is None:
                        first_content_at = now
                    content_parts.append(delta_content)
                delta_reasoning = extract_reasoning_text(chunk_dict)
                if delta_reasoning:
                    reasoning_parts.append(delta_reasoning)
                if chunk_dict.get("usage"):
                    usage_dict = normalize_usage(chunk_dict.get("usage"))
        except Exception:
            if not FALLBACK_TO_NON_STREAMING:
                raise
            payload = build_payload(system_prompt, user_prompt, cfg, stream=False)
            response = client.chat(payload)
            response_dict = obj_to_dict(response) or {}
            usage_dict = normalize_usage(response_dict.get("usage"))
            content = extract_content(response_dict)
            reasoning = extract_reasoning_text(response_dict)
            content_parts = [content] if content else []
            reasoning_parts = [reasoning] if reasoning else []
    else:
        payload = build_payload(system_prompt, user_prompt, cfg, stream=False)
        response = client.chat(payload)
        response_dict = obj_to_dict(response) or {}
        usage_dict = normalize_usage(response_dict.get("usage"))
        content = extract_content(response_dict)
        reasoning = extract_reasoning_text(response_dict)
        content_parts = [content] if content else []
        reasoning_parts = [reasoning] if reasoning else []

    end = time.perf_counter()
    answer = "".join(content_parts)
    reasoning_text = "".join(reasoning_parts)
    finish_reason = extract_finish_reason(response_dict) or extract_finish_reason(last_chunk_dict)
    usage_numbers = extract_usage_numbers(usage_dict)
    return {
        "status": "success", "error_type": None, "error_message": None,
        "total_latency_sec": end - start,
        "ttfc_sec": None if first_chunk_at is None else first_chunk_at - start,
        "ttft_sec": None if first_content_at is None else first_content_at - start,
        "generation_after_first_token_sec": None if first_content_at is None else end - first_content_at,
        "chunks_seen": chunks_seen,
        "finish_reason": finish_reason,
        "answer": answer,
        "answer_preview": answer[:ANSWER_PREVIEW_CHARS],
        "answer_chars": len(answer),
        "answer_words": count_words(answer),
        "answer_hash": text_hash(answer),
        "reasoning_text": reasoning_text,
        "reasoning_chars": len(reasoning_text),
        "raw_usage_json": json.dumps(usage_dict, ensure_ascii=False),
        **usage_numbers,
    }


def run_with_retries(row: pd.Series, cfg: Dict[str, Any]) -> Dict[str, Any]:
    last = None
    for attempt in range(1, MAX_RETRIES + 2):
        start = time.perf_counter()
        try:
            result = run_payload(row["system_prompt"], row["user_prompt"], cfg)
            result["attempts"] = attempt
            return result
        except Exception as exc:
            last = {
                "status": "error", "error_type": type(exc).__name__, "error_message": str(exc),
                "total_latency_sec": time.perf_counter() - start, "ttfc_sec": None, "ttft_sec": None,
                "generation_after_first_token_sec": None, "chunks_seen": 0, "finish_reason": None,
                "answer": "", "answer_preview": "", "answer_chars": 0, "answer_words": 0,
                "answer_hash": text_hash(""), "reasoning_text": "", "reasoning_chars": 0,
                "raw_usage_json": "{}", "prompt_tokens": None, "completion_tokens": None,
                "total_tokens": None, "precached_prompt_tokens": None, "reasoning_tokens": None,
                "attempts": attempt,
            }
            if attempt <= MAX_RETRIES:
                time.sleep(RETRY_BACKOFF_SEC * attempt + random.random() * JITTER_SEC)
    return last


In [ ]:
# =========================
# 7. Select benchmark rows and build jobs
# =========================


def select_rows(df: pd.DataFrame) -> pd.DataFrame:
    if RUN_MODE == "full":
        selected = df.copy()
    elif RUN_MODE == "sample":
        selected = df.sample(min(SAMPLE_ROWS_TOTAL, len(df)), random_state=RANDOM_SEED).copy()
    elif RUN_MODE == "smoke":
        selected = pd.concat([p.sort_values("input_chars").head(SMOKE_ROWS_PER_STAGE) for _, p in df.groupby("type", dropna=False)], ignore_index=True)
    else:
        raise ValueError(RUN_MODE)
    return selected.sort_values(["type", "input_chars", "dataset_row_id"]).reset_index(drop=True)

df_run = select_rows(df_inputs)
print(f"RUN_MODE={RUN_MODE}; rows={len(df_run)}; configs={len(df_configs)}; expected calls={len(df_run) * len(df_configs)}")
display(df_run.groupby("type").agg(rows=("dataset_row_id", "count"), min_chars=("input_chars", "min"), max_chars=("input_chars", "max")).reset_index())


def ordered_configs_for_question(question_order_index: int, dataset_row_id: int) -> List[Dict[str, Any]]:
    cfgs = [r.to_dict() for _, r in df_configs.iterrows()]
    if CONFIG_ORDER_STRATEGY == "fixed":
        return cfgs
    if CONFIG_ORDER_STRATEGY == "rotate":
        shift = question_order_index % len(cfgs)
        return cfgs[shift:] + cfgs[:shift]
    if CONFIG_ORDER_STRATEGY == "shuffle_by_question":
        rng = random.Random(RANDOM_SEED + int(dataset_row_id))
        rng.shuffle(cfgs)
        return cfgs
    raise ValueError(CONFIG_ORDER_STRATEGY)


def build_jobs() -> List[Dict[str, Any]]:
    completed = set()
    if RAW_JSONL_PATH.exists():
        with RAW_JSONL_PATH.open("r", encoding="utf-8") as f:
            for line in f:
                try:
                    item = json.loads(line)
                    completed.add((int(item["dataset_row_id"]), item["config_id"]))
                except Exception:
                    pass
    jobs = []
    global_order = 0
    for q_idx, (_, row) in enumerate(df_run.iterrows()):
        for c_idx, cfg in enumerate(ordered_configs_for_question(q_idx, int(row["dataset_row_id"]))):
            key = (int(row["dataset_row_id"]), cfg["config_id"])
            if key not in completed:
                jobs.append({"row": row, "cfg": cfg, "question_order_index": q_idx, "config_order_within_question": c_idx, "global_job_order": global_order})
            global_order += 1
    return jobs


def group_jobs_by_question(jobs: List[Dict[str, Any]]) -> List[List[Dict[str, Any]]]:
    groups, batch, current_q = [], [], None
    for job in jobs:
        q = job["question_order_index"]
        if current_q is None:
            current_q = q
        if q != current_q:
            groups.append(batch)
            batch = []
            current_q = q
        batch.append(job)
    if batch:
        groups.append(batch)
    return groups


In [ ]:
# =========================
# 8. Record, summaries, checkpoint writer
# =========================


def build_record(job: Dict[str, Any], result: Dict[str, Any]) -> Dict[str, Any]:
    row, cfg = job["row"], job["cfg"]
    rec = {
        "run_started_at": RUN_STARTED_AT,
        "run_mode": RUN_MODE,
        "config_order_strategy": CONFIG_ORDER_STRATEGY,
        "global_job_order": int(job["global_job_order"]),
        "question_order_index": int(job["question_order_index"]),
        "config_order_within_question": int(job["config_order_within_question"]),
        "dataset_row_id": int(row["dataset_row_id"]),
        "question_run_id": row.get("question_run_id"),
        "type": row.get("type"),
        "stage": row.get("stage"),
        "question": row.get("question"),
        "source_file": row.get("source_file"),
        "source_line": row.get("source_line"),
        "system_chars": int(row.get("system_chars", 0)),
        "user_chars": int(row.get("user_chars", 0)),
        "input_chars": int(row.get("input_chars", 0)),
        "old_output_chars": int(row.get("old_output_chars", 0)),
    }
    for k in ["config_id", "model_key", "model", "reasoning_mode", "reasoning_control", "max_tokens", "temperature", "top_p", "repetition_penalty", "extra_params_json", "why"]:
        rec[k] = cfg.get(k)
    rec.update(result)
    rec["empty_answer"] = (rec.get("answer_chars") or 0) == 0
    rec["possibly_truncated"] = str(rec.get("finish_reason", "")).lower() in {"length", "max_tokens", "token_limit"} or (
        pd.notna(rec.get("completion_tokens")) and pd.notna(rec.get("max_tokens")) and float(rec.get("completion_tokens")) >= 0.95 * float(rec.get("max_tokens"))
    )
    rec["ok_nonempty"] = rec.get("status") == "success" and not rec["empty_answer"]
    rec["error_or_empty"] = not rec["ok_nonempty"]
    if not SAVE_FULL_ANSWERS_TO_EXCEL:
        rec["answer"] = ""
    return rec


def execute_job(job: Dict[str, Any]) -> Dict[str, Any]:
    return build_record(job, run_with_retries(job["row"], job["cfg"]))


def read_raw_jsonl(path: Path) -> pd.DataFrame:
    records = []
    if path.exists():
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    try:
                        records.append(json.loads(line))
                    except Exception:
                        pass
    return pd.DataFrame(records)


def q(s, p):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return np.nan if s.empty else float(s.quantile(p))


def summarize_results(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    for col in ["total_latency_sec", "ttft_sec", "completion_tokens", "prompt_tokens", "total_tokens", "reasoning_tokens", "answer_chars", "answer_words", "input_chars", "max_tokens"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    for col in ["ok_nonempty", "empty_answer", "possibly_truncated", "error_or_empty"]:
        if col in df.columns:
            df[col] = df[col].fillna(False).astype(bool)

    summary = df.groupby(["config_id", "model_key", "model", "reasoning_mode", "reasoning_control", "max_tokens", "temperature", "top_p", "repetition_penalty", "extra_params_json"], dropna=False).agg(
        calls=("dataset_row_id", "count"),
        success_nonempty=("ok_nonempty", "sum"),
        errors_or_empty=("error_or_empty", "sum"),
        empty_answers=("empty_answer", "sum"),
        possible_truncations=("possibly_truncated", "sum"),
        avg_total_latency_sec=("total_latency_sec", "mean"),
        p50_total_latency_sec=("total_latency_sec", lambda s: q(s, .5)),
        p90_total_latency_sec=("total_latency_sec", lambda s: q(s, .9)),
        avg_ttft_sec=("ttft_sec", "mean"),
        avg_completion_tokens=("completion_tokens", "mean"),
        avg_total_tokens=("total_tokens", "mean"),
        avg_reasoning_tokens=("reasoning_tokens", "mean"),
        avg_answer_chars=("answer_chars", "mean"),
        avg_answer_words=("answer_words", "mean"),
        avg_reasoning_chars=("reasoning_chars", "mean"),
    ).reset_index()
    summary["success_rate_pct"] = 100 * summary["success_nonempty"] / summary["calls"]
    summary["empty_answer_rate_pct"] = 100 * summary["empty_answers"] / summary["calls"]
    summary["possible_truncation_rate_pct"] = 100 * summary["possible_truncations"] / summary["calls"]
    summary = summary.round(3)

    by_stage = df.groupby(["config_id", "model_key", "reasoning_mode", "type"], dropna=False).agg(
        calls=("dataset_row_id", "count"),
        success_nonempty=("ok_nonempty", "sum"),
        avg_total_latency_sec=("total_latency_sec", "mean"),
        p90_total_latency_sec=("total_latency_sec", lambda s: q(s, .9)),
        avg_answer_words=("answer_words", "mean"),
        possible_truncations=("possibly_truncated", "sum"),
        empty_answers=("empty_answer", "sum"),
    ).reset_index()
    by_stage["success_rate_pct"] = 100 * by_stage["success_nonempty"] / by_stage["calls"]
    by_stage["possible_truncation_rate_pct"] = 100 * by_stage["possible_truncations"] / by_stage["calls"]
    by_stage["empty_answer_rate_pct"] = 100 * by_stage["empty_answers"] / by_stage["calls"]
    by_stage = by_stage.round(3)

    rank = summary.copy()
    rank["latency_rank"] = rank["p90_total_latency_sec"].rank(method="min", ascending=True)
    rank["success_rank"] = rank["success_rate_pct"].rank(method="min", ascending=False)
    rank["truncation_rank"] = rank["possible_truncation_rate_pct"].rank(method="min", ascending=True)
    rank["empty_rank"] = rank["empty_answer_rate_pct"].rank(method="min", ascending=True)
    rank["rank_score"] = 0.45 * rank["latency_rank"] + 0.30 * rank["success_rank"] + 0.15 * rank["truncation_rank"] + 0.10 * rank["empty_rank"]
    rank = rank.sort_values(["rank_score", "success_rate_pct", "p90_total_latency_sec"], ascending=[True, False, True]).reset_index(drop=True)
    return summary, by_stage, rank


def write_excel_report(path: Path, reason: str, completed_question_groups: Optional[int] = None) -> None:
    raw_df = read_raw_jsonl(RAW_JSONL_PATH)
    summary, by_stage, rank = summarize_results(raw_df.copy()) if not raw_df.empty else (pd.DataFrame(), pd.DataFrame(), pd.DataFrame())
    readme = pd.DataFrame([
        ["run_started_at", RUN_STARTED_AT],
        ["report_reason", reason],
        ["completed_question_groups_in_this_session", completed_question_groups],
        ["run_mode", RUN_MODE],
        ["raw_records_saved", len(raw_df)],
        ["configs", len(df_configs)],
        ["checkpoint_every_questions", CHECKPOINT_EVERY_QUESTIONS],
        ["job_order", "question-major"],
        ["raw_jsonl", str(RAW_JSONL_PATH)],
    ], columns=["field", "value"])

    raw_excel = raw_df.copy()
    if "answer" in raw_excel.columns:
        raw_excel["answer"] = raw_excel["answer"].fillna("").str.slice(0, 30000)
    if "reasoning_text" in raw_excel.columns:
        raw_excel["reasoning_text"] = raw_excel["reasoning_text"].fillna("").str.slice(0, 5000)

    with pd.ExcelWriter(path, engine="xlsxwriter") as writer:
        readme.to_excel(writer, sheet_name="README", index=False)
        df_available_parameters.to_excel(writer, sheet_name="AvailableParameters", index=False)
        df_configs.to_excel(writer, sheet_name="Configs", index=False)
        df_run.to_excel(writer, sheet_name="InputRows", index=False)
        rank.to_excel(writer, sheet_name="RankedConfigs", index=False)
        summary.to_excel(writer, sheet_name="SummaryOverall", index=False)
        by_stage.to_excel(writer, sheet_name="SummaryByStage", index=False)
        raw_excel.to_excel(writer, sheet_name="RawResults", index=False)
        workbook = writer.book
        header_fmt = workbook.add_format({"bold": True, "bg_color": "#D9EAF7", "border": 1})
        wrap_fmt = workbook.add_format({"text_wrap": True, "valign": "top"})
        for sheet_name in writer.sheets:
            ws = writer.sheets[sheet_name]
            ws.freeze_panes(1, 0)
            ws.set_row(0, None, header_fmt)
            ws.set_column(0, 12, 20)
            ws.set_column(12, 80, 28, wrap_fmt)
    print("Excel saved:", path)


In [ ]:
# =========================
# 9. Execute benchmark with checkpoint overwrite every 3 questions
# =========================

if RUN_BENCHMARK:
    jobs = build_jobs()
    question_batches = group_jobs_by_question(jobs)
    print("Jobs to run:", len(jobs))
    print("Question groups to run:", len(question_batches))
    print(f"Checkpoint overwrites {CHECKPOINT_LATEST_PATH.name} every {CHECKPOINT_EVERY_QUESTIONS} question groups.")
    all_new = []
    completed_question_groups = 0
    pbar = None
    try:
        with RAW_JSONL_PATH.open("a", encoding="utf-8") as out_f:
            pbar = tqdm(total=len(jobs), desc="Benchmark calls")
            for batch in question_batches:
                if MAX_WORKERS == 1:
                    batch_records = [execute_job(job) for job in batch]
                else:
                    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                        futures = [executor.submit(execute_job, job) for job in batch]
                        batch_records = [fut.result() for fut in as_completed(futures)]
                # Пишем только завершенную группу вопроса целиком.
                for rec in batch_records:
                    out_f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                    out_f.flush()
                    all_new.append(rec)
                    pbar.update(1)
                completed_question_groups += 1
                if CHECKPOINT_EVERY_QUESTIONS and completed_question_groups % CHECKPOINT_EVERY_QUESTIONS == 0:
                    write_excel_report(CHECKPOINT_LATEST_PATH, reason="periodic_checkpoint", completed_question_groups=completed_question_groups)
            if pbar is not None:
                pbar.close()
    except KeyboardInterrupt:
        print("Interrupted. Writing latest checkpoint before stopping...")
        write_excel_report(CHECKPOINT_LATEST_PATH, reason="keyboard_interrupt", completed_question_groups=completed_question_groups)
        raise
    finally:
        if pbar is not None:
            pbar.close()
        # Даже если новых строк в этой сессии не было, checkpoint полезен как актуальный срез raw_results.jsonl.
        write_excel_report(CHECKPOINT_LATEST_PATH, reason="latest_after_benchmark_cell", completed_question_groups=completed_question_groups)
    print("New records:", len(all_new))
else:
    print("RUN_BENCHMARK=False: benchmark skipped")


In [ ]:
# =========================
# 10. Final Excel report and quick view
# =========================

write_excel_report(EXCEL_REPORT_PATH, reason="final_report")
raw_df = read_raw_jsonl(RAW_JSONL_PATH)
summary, by_stage, rank = summarize_results(raw_df.copy()) if not raw_df.empty else (pd.DataFrame(), pd.DataFrame(), pd.DataFrame())
print("Final report:", EXCEL_REPORT_PATH.resolve())
print("Latest checkpoint:", CHECKPOINT_LATEST_PATH.resolve())
print("Raw records:", len(raw_df))
if not rank.empty:
    display(rank.head(12))
